# 3 — Ask what a valid waveform really means

**Learning objective:** distinguish file integrity, structural validity and clinical
signal quality, then inspect how a PyTorch Dataset supplies consistent signal batches.
This tutorial has no clinical labels, train/test splits, loss functions or classifiers.

We use the bundled public sample and the real implementations in
[`catalog.py`](../src/ecg_waveform_lab/catalog.py) and
[`signal.py`](../src/ecg_waveform_lab/signal.py). Our path is **verify bytes → inspect
signal properties → understand explicit failure → use Dataset and DataLoader**.
A handful of checks in this notebook teach the concepts; the automated tests remain
in [`tests/`](../tests/) and are run separately.

## Setup: use this checkout's installed package

Run `uv sync --locked` and launch `uv run --frozen jupyter lab notebooks` from the
repository root. Choose its Python 3 kernel and run this notebook in order.
Each tutorial works independently. The installed package locates the bundled
sample relative to the checkout; there are no machine-specific paths or downloads.
The environment versions below make your numerical session inspectable.

In [ ]:
import os
from importlib.metadata import version

from ecg_waveform_lab import project_root

root = project_root()
os.environ.setdefault("MPLCONFIGDIR", str(root / ".cache/matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from ecg_waveform_lab import (
    CANONICAL_LEADS,
    calibrate,
    collate_windows,
    load_catalog,
    make_window,
    prepare_ecg,
    read_native,
    reorder_leads,
    resample_signal,
    verify_sample,
    WaveformDataset,
)

display(pd.Series({
    package: version(package)
    for package in ("numpy", "scipy", "wfdb", "torch", "matplotlib")
}, name="Installed version"))

**What we learned:** this notebook is independent and reads only the bundled public
examples. It does not require a research manifest or generated clinical metadata.

## 3.1 Separate checksum integrity from clinical validity

A SHA-256 hash identifies bytes. Matching the publisher's checksum establishes that
our copy matches the released file; it does not prove correct electrode placement,
correct labels or absence of acquisition artefact. The acquisition process pinned
the full official checksum list; offline verification uses its retained excerpt
and the sample manifest. It reads all 24 bundled native files, not a large dataset.

In [ ]:
integrity_result = verify_sample(root)
print(integrity_result)

The public sample was selected with a fixed seed and engineering strata, not as a
representative clinical cohort. Inspect its rates, durations and reasons for inclusion.
These public record IDs are filenames, not a new patient linkage table. We do not
extract demographic or diagnosis comments into additional metadata.

In [ ]:
catalog = load_catalog(root)
display(pd.DataFrame([
    {
        "public_record": record.record_id,
        "source_cohort": record.cohort,
        "native_Hz": record.sample_rate_hz,
        "duration_s": record.duration_seconds,
        "selection_reason": record.selection_reason,
    }
    for record in catalog
]))

**What we learned:** provenance and purpose are part of reproducibility. This sample
can demonstrate signal engineering, but cannot estimate clinical prevalence or model
performance. Its source files are unchanged from the official release.

## 3.2 Inspect the signal before summarising it

Choose the unusual 257 Hz INCART example. `read_native` exposes its original shape
and channel order. `prepare_ecg` performs canonical ordering, full-record resampling
and the ten-second window. It neither normalises amplitudes nor applies a quality
threshold. The full native recording is much longer than the displayed window.

In [ ]:
record = next(item for item in catalog if item.sample_rate_hz == 257)
native = read_native(record.header_path)
window = prepare_ecg(native)
print("Native array:", native.millivolts.shape, "at", native.sample_rate_hz, "Hz")
print(
    "Prepared tensor:", tuple(window.waveform.shape),
    "at", window.sample_rate_hz, "Hz",
)
print("Tensor units:", window.units)
print("Transformation flags:", window.flags)

Peak-to-peak range is maximum minus minimum. An exactly zero range flags a constant
channel, but a nonzero range does not prove a usable lead. Saturation, noise,
reversed electrodes and baseline drift require more nuanced analysis. The following
table describes the first ten-second tensor without imposing a clinical cutoff.

In [ ]:
window_values = window.waveform.numpy()
lead_summary = pd.DataFrame({
    "lead": CANONICAL_LEADS,
    "minimum_mV": window_values.min(axis=1),
    "maximum_mV": window_values.max(axis=1),
    "peak_to_peak_mV": np.ptp(window_values, axis=1),
})
display(lead_summary.set_index("lead").round(4))
print("Finite values:", bool(torch.isfinite(window.waveform).all()))
print("Supported time samples:", int(window.valid_time_mask.sum()))

**Try changing** the displayed lead or zoom duration. Compare native and prepared
values on a common time axis, with their original units. The denser 500 Hz grid
interpolates from 257 Hz measurements; it does not add independent observations.

In [ ]:
lead_name = "V1"
view_seconds = 2.0
native_channel = native.lead_names.index(lead_name)
output_channel = CANONICAL_LEADS.index(lead_name)
native_count = round(view_seconds * native.sample_rate_hz)
output_count = round(view_seconds * window.sample_rate_hz)
native_time = np.arange(native_count) / native.sample_rate_hz
output_time = np.arange(output_count) / window.sample_rate_hz

figure, axis = plt.subplots(figsize=(11, 4), constrained_layout=True)
axis.plot(
    native_time,
    native.millivolts[native_channel, :native_count],
    label="Native 257 Hz",
)
axis.plot(
    output_time, window_values[output_channel, :output_count],
    linestyle="--", label="Prepared 500 Hz",
)
axis.set(
    xlabel="Time (s)",
    ylabel="Voltage (mV)",
    title=f"Public INCART example, lead {lead_name}",
)
axis.legend()
axis.grid(alpha=0.2)
plt.show()

**What we learned:** a valid shape, finite numbers and plausible-looking traces
answer limited engineering questions. They are not interchangeable with clinical
signal quality or reliable diagnosis annotations.

## 3.3 Explore failure without modifying a file

Choose `failure_kind` from `"nan"`, `"missing_lead"` or `"flat"`. We make a clearly
labelled **in-memory perturbation** of the decoded object, leaving the original
recording untouched. Predict the outcome: a nonfinite value or missing lead must
raise; a constant lead is reported as a flag by this educational package.

That distinction is deliberate: malformed inputs cannot satisfy the tensor
contract, while an exact flatline is surfaced for inspection without claiming a
clinically validated rejection threshold. No failed recording is silently repaired.

In [ ]:
from dataclasses import replace

failure_kind = "nan"
perturbed_values = native.millivolts.copy()
perturbed_names = native.lead_names
if failure_kind == "nan":
    perturbed_values[0, 0] = np.nan
elif failure_kind == "missing_lead":
    perturbed_names = native.lead_names[:-1]
elif failure_kind == "flat":
    perturbed_values[1, :] = 0.0
else:
    raise ValueError("Choose nan, missing_lead or flat")

perturbed = replace(
    native, millivolts=perturbed_values, lead_names=perturbed_names
)
print("Original and altered arrays share memory:", np.shares_memory(
    native.millivolts, perturbed.millivolts
))

Now call the same preparation function as the real path. The exception handler is
limited to the deliberately requested invalid cases. Unexpected failure of the flat
case remains an error, and unexpected acceptance of malformed input also fails.

In [ ]:
try:
    demonstration_window = prepare_ecg(perturbed)
except ValueError as error:
    if failure_kind == "flat":
        raise
    print("Expected rejection:", str(error))
else:
    if failure_kind != "flat":
        raise AssertionError("Malformed input should have been rejected")
    print("Explicit flags:", demonstration_window.flags)

**What we learned:** error handling is part of the interface. We have demonstrated
rejection and flagging, not changed a source file or invented a replacement ECG.

## 3.4 Inspect the complete small sample without hiding failures

Twelve examples are small enough for a local demonstration. We retain one status
row per public recording, including an error if any occurs. Success is not a
population-wide quality claim. The loop checks structure through the package and
summarises prepared windows; it does not silently discard a failed record.

In [ ]:
status_rows = []
for sample_record in catalog:
    try:
        sample_native = read_native(sample_record.header_path)
        sample_window = prepare_ecg(sample_native)
    except ValueError as error:
        status_rows.append({
            "public_record": sample_record.record_id,
            "status": "failed",
            "detail": str(error),
        })
    else:
        status_rows.append({
            "public_record": sample_record.record_id,
            "status": "decoded and prepared",
            "detail": ", ".join(sample_window.flags) or "no transformation flags",
        })

display(pd.DataFrame(status_rows))
failures = [row for row in status_rows if row["status"] == "failed"]
print("Failed examples:", len(failures))

**What we learned:** a transparent status table preserves failures rather than
making a batch appear clean through silent replacement. Full engineering regression
checks, including comparison with WFDB for every bundled example, live in the tests.

## 3.5 Let Dataset and DataLoader define the batch boundary

A Dataset holds references and loads an example when indexed. A DataLoader obtains
examples and calls a collator. Here we choose the first four catalog entries in
catalog order, with zero workers and no shuffling so the sequence is visible.
`collate_windows` checks shape, units, lead order, rate and finite values.
There are no targets, supervision masks or training partitions in this project.

In [ ]:
from torch.utils.data import DataLoader

batch_size = 4
selected_records = catalog[:batch_size]
dataset = WaveformDataset(selected_records)
loader = DataLoader(
    dataset,
    batch_size=batch_size,
    num_workers=0,
    shuffle=False,
    collate_fn=collate_windows,
)
print("Dataset references:", len(dataset))

Fetching a batch triggers actual reads. The output's record order must match the
selected references; the padding mask has one row per example. Try another batch
size from 1 to 12. Do not shuffle arrays and identities separately.

In [ ]:
batch = next(iter(loader))
print("Waveform:", tuple(batch["waveform"].shape), batch["waveform"].dtype)
print("Time mask:", tuple(batch["valid_time_mask"].shape))
print("Units:", batch["units"])
print("Record order preserved:", batch["record_ids"] == [
    item.record_id for item in selected_records
])
assert batch["waveform"].shape == (len(selected_records), 12, 5000)
assert torch.isfinite(batch["waveform"]).all()

**What we learned:** the sample can supply consistent CPU float32 tensors through a
normal PyTorch data interface. This is useful groundwork for signal processing and
future ML studies, but this repository does not implement model training, clinical
labels, diagnosis classification or performance evaluation.

**Explain back:** what can a checksum establish? Can a correctly decoded ECG have
poor clinical quality? Why is zero padding different from a measured flatline?
Why must identities stay aligned with the example axis?

You have now traced stored values through calibration, lead ordering, sample-rate
conversion, windowing, masks and batching. Consult [the signal contract](../docs/SIGNAL_CONTRACT.md)
and run `uv run --frozen python -m unittest discover -s tests -v` for the separate
engineering checks. Data credits and modifications are in [ATTRIBUTION.md](../data/ATTRIBUTION.md).